# IAU 140
Hugo Foltin, Teodor Potančok

In [7]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 1.1 Základný opis dát spolu s ich charakteristikami


### A
* V tejto časti načítame všetky tri vstupné dátové súbory, zistíme ich štruktúru, rozmery, dátové typy, pamäťovú náročnosť a výskyt chýbajúcich hodnôt 

In [8]:
units = pd.read_csv("data/units.csv", sep=",")
readings = pd.read_csv("data/readings.csv", sep="\t")
gateways = pd.read_csv("data/gateways.csv", sep=";")

print("SÚHRN ROZMEROV DÁTOVÝCH SAD")
print(f"units.csv:    {units.shape[0]:>6} riadkov, {units.shape[1]:>2} stĺpcov")
print(f"readings.csv: {readings.shape[0]:>6} riadkov, {readings.shape[1]:>2} stĺpcov")
print(f"gateways.csv: {gateways.shape[0]:>6} riadkov, {gateways.shape[1]:>2} stĺpcov")

SÚHRN ROZMEROV DÁTOVÝCH SAD
units.csv:      2000 riadkov,  7 stĺpcov
readings.csv:  12313 riadkov, 15 stĺpcov
gateways.csv:  16000 riadkov,  8 stĺpcov


In [9]:
def analyze_structure(df, name):
    summary = pd.DataFrame({
        "Dátový typ": df.dtypes.astype(str),
        "Non-Null Počet": df.notnull().sum(),
        "Null Počet": df.isnull().sum(),
        "Null Podiel (%)": (df.isnull().sum() / len(df) * 100).round(2),
        "Počet unikátnych": df.nunique(),
        "Ukážka hodnoty": df.iloc[0].values
    })
    print(f"\nŠtruktúra tabuľky: {name} ({df.shape[0]} riadkov, {df.shape[1]} stĺpcov, {df.memory_usage().sum() / 1024:.1f} KB)")
    return summary

print("1. UNITS:")
display(analyze_structure(units, "units.csv"))

print("\n2. READINGS:")
display(analyze_structure(readings, "readings.csv"))

print("\n3. GATEWAYS:")
display(analyze_structure(gateways, "gateways.csv"))

1. UNITS:

Štruktúra tabuľky: units.csv (2000 riadkov, 7 stĺpcov, 109.5 KB)


,Dátový typ,Non-Null Počet,Null Počet,Null Podiel (%),Počet unikátnych,Ukážka hodnoty
depot_id,int64,2000,0,0.0,120,12
code_fleet,str,2000,0,0.0,5,FB
id_unit,int64,2000,0,0.0,2000,1
duty_class,str,2000,0,0.0,20,1
service_hours,float64,1900,100,5.0,1871,8329.4
install_region,str,2000,0,0.0,58,Fürstenfeldbruck
codeModel,str,2000,0,0.0,4,XC-7



2. READINGS:

Štruktúra tabuľky: readings.csv (12313 riadkov, 15 stĺpcov, 1443.1 KB)


,Dátový typ,Non-Null Počet,Null Počet,Null Podiel (%),Počet unikátnych,Ukážka hodnoty
id_unit,str,12313,0,0.00,2111,U01083
cycleTs,str,12313,0,0.00,4065,"06/22/2023, 13:51:00"
motorCurrent,float64,12313,0,0.00,12007,4.10064
filterDp,float64,11782,531,4.31,11584,NaN
oil_temperature,float64,12313,0,0.00,12080,81.25839
reservoirPressure,float64,12313,0,0.00,11794,8.66503
pressure_working,float64,12313,0,0.00,11771,8.55913
oilPressure,float64,12313,0,0.00,11768,3.71709
vibrationRms,float64,11799,514,4.17,11447,2.11896
flowmeter,float64,12313,0,0.00,12056,30.07764



3. GATEWAYS:

Štruktúra tabuľky: gateways.csv (16000 riadkov, 8 stĺpcov, 1000.1 KB)


,Dátový typ,Non-Null Počet,Null Počet,Null Podiel (%),Počet unikátnych,Ukážka hodnoty
gatewayId,str,16000,0,0.0,16000,GW-00000
depot_id,int64,16000,0,0.0,120,17
installDate,str,16000,0,0.0,9669,2023/05/29
mac_address,str,16000,0,0.0,16000,66:3a:5c:ca:d9:c6
id_unit,float64,2000,14000,87.5,2000,1525.0
Lon,float64,16000,0,0.0,15594,-8.18846
versionFirmware,str,16000,0,0.0,3,v2.7.0
model_hardware,str,16000,0,0.0,5,GWX-300


Pre každý dataset skontrolujeme typy a počet záznamov

In [4]:
units.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   depot_id        2000 non-null   int64  
 1   code_fleet      2000 non-null   str    
 2   id_unit         2000 non-null   int64  
 3   duty_class      2000 non-null   str    
 4   service_hours   1900 non-null   float64
 5   install_region  2000 non-null   str    
 6   codeModel       2000 non-null   str    
dtypes: float64(1), int64(2), str(4)
memory usage: 109.5 KB


In [5]:
readings.info()

<class 'pandas.DataFrame'>
RangeIndex: 12313 entries, 0 to 12312
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_unit            12313 non-null  str    
 1   cycleTs            12313 non-null  str    
 2   motorCurrent       12313 non-null  float64
 3   filterDp           11782 non-null  float64
 4   oil_temperature    12313 non-null  float64
 5   reservoirPressure  12313 non-null  float64
 6   pressure_working   12313 non-null  float64
 7   oilPressure        12313 non-null  float64
 8   vibrationRms       11799 non-null  float64
 9   flowmeter          12313 non-null  float64
 10  humidityIntake     11714 non-null  float64
 11  ambient_noise      12313 non-null  float64
 12  dischargeDewpoint  12313 non-null  float64
 13  current_aux        12313 non-null  float64
 14  Failure            12313 non-null  int64  
dtypes: float64(12), int64(1), str(2)
memory usage: 1.4 MB


In [6]:
gateways.info()

<class 'pandas.DataFrame'>
RangeIndex: 16000 entries, 0 to 15999
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   gatewayId        16000 non-null  str    
 1   depot_id         16000 non-null  int64  
 2   installDate      16000 non-null  str    
 3   mac_address      16000 non-null  str    
 4   id_unit          2000 non-null   float64
 5   Lon              16000 non-null  float64
 6   versionFirmware  16000 non-null  str    
 7   model_hardware   16000 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 1000.1 KB


## Čistenie datasetu

Vidíme že polia v datasete majú nekonzistentné názvy - `depot_id` vs `id_unit`, `versionFirmware` vs `model_hardware`. Aby sa s dátami lepšie pracovalo musíme ich premenovať aby používali jednu konvenciu. V Pythone sa štandardne používa `camel_case` preto aj my prejdeme na tento štýl.

In [7]:
units.rename(
    columns={
        "code_fleet": "fleet_code",
        "id_unit": "unit_id",
        "codeModel": "model_code",
    },
    inplace=True,
)
units.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   depot_id        2000 non-null   int64  
 1   fleet_code      2000 non-null   str    
 2   unit_id         2000 non-null   int64  
 3   duty_class      2000 non-null   str    
 4   service_hours   1900 non-null   float64
 5   install_region  2000 non-null   str    
 6   model_code      2000 non-null   str    
dtypes: float64(1), int64(2), str(4)
memory usage: 109.5 KB


In [8]:
readings.rename(
    columns={
        "id_unit": "unit_id",
        "cycleTs": "cycle_timestamp",
        "motorCurrent": "motor_current",
        "filterDp": "filter_dp",
        "reservoirPressure": "reservoir_pressure",
        "pressure_working": "working_pressure",
        "oilPressure": "oil_pressure",
        "vibrationRms": "vibration_rms",
        "flowmeter": "flow_meter",
        "humidityIntake": "humidity_intake",
        "dischargeDewpoint": "discharge_dewpoint",
        "current_aux": "aux_current",
        "Failure": "is_failing",
    },
    inplace=True,
)
readings.info()

<class 'pandas.DataFrame'>
RangeIndex: 12313 entries, 0 to 12312
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   unit_id             12313 non-null  str    
 1   cycle_timestamp     12313 non-null  str    
 2   motor_current       12313 non-null  float64
 3   filter_dp           11782 non-null  float64
 4   oil_temperature     12313 non-null  float64
 5   reservoir_pressure  12313 non-null  float64
 6   working_pressure    12313 non-null  float64
 7   oil_pressure        12313 non-null  float64
 8   vibration_rms       11799 non-null  float64
 9   flow_meter          12313 non-null  float64
 10  humidity_intake     11714 non-null  float64
 11  ambient_noise       12313 non-null  float64
 12  discharge_dewpoint  12313 non-null  float64
 13  aux_current         12313 non-null  float64
 14  is_failing          12313 non-null  int64  
dtypes: float64(12), int64(1), str(2)
memory usage: 1.4 MB


In [9]:
gateways.rename(
    columns={
        "gatewayId": "gateway_id",
        "installDate": "install_date",
        "id_unit": "unit_id",
        "Lon": "longitude",
        "versionFirmware": "firmware_version",
        "model_hardware": "hardware_model",
    },
    inplace=True,
)
gateways.info()

<class 'pandas.DataFrame'>
RangeIndex: 16000 entries, 0 to 15999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gateway_id        16000 non-null  str    
 1   depot_id          16000 non-null  int64  
 2   install_date      16000 non-null  str    
 3   mac_address       16000 non-null  str    
 4   unit_id           2000 non-null   float64
 5   longitude         16000 non-null  float64
 6   firmware_version  16000 non-null  str    
 7   hardware_model    16000 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 1000.1 KB


Mimo toho vidíme že v `gateways` je veľa záznamov v ktorých nemáme kompletné údaje - chýba `filter_dp`, `vibration_rms` alebo `humidity_intake`. Pre začiatok zistíme koľko ich je:

In [11]:
readings.isna().sum()

unit_id                 0
cycle_timestamp         0
motor_current           0
filter_dp             531
oil_temperature         0
reservoir_pressure      0
working_pressure        0
oil_pressure            0
vibration_rms         514
flow_meter              0
humidity_intake       599
ambient_noise           0
discharge_dewpoint      0
aux_current             0
is_failing              0
dtype: int64

In [13]:
readings.dropna().shape[0] / readings.shape[0] * 100

87.55786567042962

Po odstránení poškodených záznamov ich stále zostane viac ako 87%, preto sa to určite oplatí a môžeme pokračovať bez nich.

In [ ]:
readings.dropna(inplace=True)

V tabuľke `gateways` je iný problém, máme 16 000 záznamov ale len 2 000 z nich má `unit_id` ktorý potrebujeme na prepojenie s `units` a `readings`. Podľa všetkého to je zoznam sieťového vybavenie (máme MAC adresy a názvy modelov ako `NODE-9`), z ktorého väčšina nesúvisí s dopravným systémom. Záznamy bez `unit_id` sú pre nás zbytočné a je lepšie ich odstrániť.

In [14]:
gateways.dropna(inplace=True)

In [15]:
gateways.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gateway_id        2000 non-null   str    
 1   depot_id          2000 non-null   int64  
 2   install_date      2000 non-null   str    
 3   mac_address       2000 non-null   str    
 4   unit_id           2000 non-null   float64
 5   longitude         2000 non-null   float64
 6   firmware_version  2000 non-null   str    
 7   hardware_model    2000 non-null   str    
dtypes: float64(2), int64(1), str(5)
memory usage: 125.1 KB
